In [ ]:
pip install medmnist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 5.6 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms
from torchvision.transforms import v2
import medmnist
from medmnist import BreastMNIST
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix, f1_score, recall_score, roc_auc_score, roc_curve
import seaborn as sns

# trasformazioni
transform_train = v2.Compose([
    v2.ToImage(),
    # data augmentation applicata solo qui
    v2.RandomRotation(10),
    v2.RandomHorizontalFlip(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5,), (0.5,))
])

transform_test = v2.Compose([
    v2.ToImage(),
    # test e validation puliti, solo normalizzazione
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5,), (0.5,))
])

batch_size = 32

# divisione del dataset in train, validation e test
trainset = BreastMNIST(split='train', transform=transform_train, download=True)
valset = BreastMNIST(split='val', transform=transform_test, download=True)
testset = BreastMNIST(split='test', transform=transform_test, download=True)

trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True)
valloader = DataLoader(valset, batch_size=batch_size, shuffle=False)
testloader = DataLoader(testset, batch_size=batch_size, shuffle=False)

classes = ('malignant', 'benign')

# definizione della rete
class CNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1, 6, 5)
    self.pool = nn.MaxPool2d(2, 2)
    self.conv2 = nn.Conv2d(6, 16, 5)
    self.fc1 = nn.Linear(16 * 4 * 4, 120)
    self.fc2 = nn.Linear(120, 84)
    self.fc3 = nn.Linear(84, 2)

  def forward(self, x):
    x = self.pool(F.relu(self.conv1(x)))
    x = self.pool(F.relu(self.conv2(x)))
    x = torch.flatten(x, 1)   # appiattisce il tensore per i layer lineari
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x

net = CNN()

# Loss pesata e ottimizzatore
class_counts = torch.tensor([147, 399], dtype=torch.float32)
total_samples = class_counts.sum()
class_weights = total_samples / class_counts
print(f"Class weights: {class_weights}")


criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)

# training loop
num_epochs = 30
best_val_loss = float('inf')

for epoch in range(num_epochs):
    net.train()
    running_loss = 0.0

    for inputs, labels in trainloader:
        labels = labels.squeeze(1).long()
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(trainloader)

    # validation loop
    net.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in valloader:
            labels = labels.squeeze(1).long()
            outputs = net(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_loss = val_loss / len(valset)
    val_acc = 100 * correct_val / total_val

    # salvataggio basato sulla minima Validation Loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(net.state_dict(), 'best_cnn.pth')
        print(f"  --> New best model saved at epoch {epoch+1}")

    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Accuracy: {val_acc:.2f}%")

# caricamento del modello migliore per il test finale
net.load_state_dict(torch.load('best_cnn.pth'))
net.eval()

dataiter = iter(testloader)
images, labels = next(dataiter)

# rimozione della normalizzazione per la visualizzazione
images_unnormalized = images * 0.5 + 0.5
plt.imshow(torchvision.utils.make_grid(images_unnormalized).permute(1, 2, 0).cpu().numpy())
plt.show()

print('GroundTruth: ', ' '.join(f'{classes[labels[j].item()]:10s}' for j in range(len(labels))))
outputs = net(images)
_, predicted = torch.max(outputs, 1)
print('Predicted:   ', ' '.join(f'{classes[predicted[j].item()]:10s}' for j in range(len(predicted))))

# test loop
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        labels = labels.squeeze(1).long()
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the {total} test images: {100 * correct / total:.2f} %')

all_predictions = []
all_labels = []

with torch.no_grad():
    for data in testloader:
        images, labels = data
        labels = labels.squeeze(1)
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

#conversione a numpy arrays
all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

# matrice di confusione
cm = confusion_matrix(all_labels, all_predictions)
print('Confusion Matrix:')
print(cm)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

# F1 Score
f1 = f1_score(all_labels, all_predictions, average='weighted')
print(f'F1 Score (weighted): {f1:.4f}')

# recall Score
recall = recall_score(all_labels, all_predictions, average='weighted')
print(f'Recall Score (weighted): {recall:.4f}')

# AUC
all_probabilities = []
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = net(images)
        probabilities = F.softmax(outputs, dim=1)[:, 1].cpu().numpy() # probabilità di positivo
        all_probabilities.extend(probabilities)

auc = roc_auc_score(all_labels, all_probabilities)
print(f'AUC Score: {auc:.4f}')

# plot ROC Curve
fpr, tpr, thresholds = roc_curve(all_labels, all_probabilities)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='orange', label=f'ROC curve (area = {auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()